In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from pathlib import Path
plt.style.use('seaborn-v0_8-whitegrid')
BIAS = Path('artifacts/bias')
Path('artifacts').mkdir(exist_ok=True)
print('Imports OK')

In [ ]:
summary    = pd.read_csv(BIAS / 'bias_summary.csv').iloc[0]
gender_df  = pd.read_csv(BIAS / 'sasrec_gender_bias.csv')
age_df     = pd.read_csv(BIAS / 'sasrec_age_bias.csv')
genre_df   = pd.read_csv(BIAS / 'sasrec_genre_concentration.csv')
pop_s      = pd.read_csv(BIAS / 'sasrec_popularity_bias.csv').iloc[0]
pop_v      = pd.read_csv(BIAS / 'svd_popularity_bias.csv').iloc[0]
print('Bias data loaded.')
print(f'SASRec popularity ratio: {summary["sasrec_popularity_ratio"]:.2f}x')
print(f'SVD popularity ratio: {summary["svd_popularity_ratio"]:.2f}x')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
models = ['SASRec', 'SVD']
ratios = [pop_s['popularity_ratio'], pop_v['popularity_ratio']]
lt_frac = [pop_s['longtail_fraction'], pop_v['longtail_fraction']]
axes[0].bar(models, ratios, color=['steelblue', 'coral'], edgecolor='white')
axes[0].axhline(1.0, color='gray', linestyle='--', linewidth=1, label='Fair baseline (1.0)')
axes[0].set_ylabel('Popularity ratio (higher = more biased)')
axes[0].set_title('Popularity Bias: SASRec vs SVD')
axes[0].legend()
for i, v in enumerate(ratios):
    axes[0].text(i, v + 0.05, f'{v:.2f}x', ha='center', fontsize=11, fontweight='bold')
axes[1].bar(models, [f*100 for f in lt_frac], color=['steelblue', 'coral'], edgecolor='white')
axes[1].set_ylabel('Long-tail fraction (%)')
axes[1].set_title('Long-Tail Coverage: SASRec vs SVD')
for i, v in enumerate(lt_frac):
    axes[1].text(i, v*100 + 0.1, f'{v:.1%}', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('artifacts/bias_popularity.png', dpi=120)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(gender_df['group'], gender_df['HR@10'],
              color=['coral', 'steelblue'], edgecolor='white')
ax.set_ylabel('HR@10 (full-catalog)')
for bar, (_, row) in zip(bars, gender_df.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{row["HR@10"]:.4f}\n(n={row["n_users"]})',
            ha='center', fontsize=10)
gap = summary['gender_gap_HR10']
ax.set_ylim(0, 0.42)
ax.set_title(f'Gender Bias \u2014 Gap (M-F): {gap:+.4f} HR@10')
plt.tight_layout()
plt.savefig('artifacts/bias_gender.png', dpi=120)
plt.show()

In [ ]:
age_order = ['<18','18-24','25-34','35-44','45-49','50-55','56+']
age_df_sorted = age_df.set_index('group').reindex(age_order).reset_index()
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(age_df_sorted['group'], age_df_sorted['HR@10'],
              color='steelblue', edgecolor='white')
ax.axhline(age_df_sorted['HR@10'].mean(), color='red', linestyle='--',
           linewidth=1, label=f'Mean HR@10 = {age_df_sorted["HR@10"].mean():.4f}')
for bar, (_, row) in zip(bars, age_df_sorted.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{row["HR@10"]:.3f}', ha='center', fontsize=8)
ax.set_ylabel('HR@10'); ax.set_xlabel('Age group')
ax.set_title(f'Age Bias \u2014 Range: {summary["age_hr10_range"]:.4f} HR@10')
ax.legend()
plt.tight_layout()
plt.savefig('artifacts/bias_age.png', dpi=120)
plt.show()
print(f'Under-18 HR@10: {age_df_sorted.iloc[0]["HR@10"]:.4f} (lowest)')
print(f'50-55  HR@10: {age_df_sorted.iloc[5]["HR@10"]:.4f} (highest)')

In [ ]:
top10 = genre_df.head(10)
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(top10['genre'][::-1], top10['fraction_in_recs'][::-1],
        color='steelblue', edgecolor='white')
ax.set_xlabel('Fraction of recommendations containing genre')
ax.set_title('Genre Concentration in SASRec Recommendations (Top 10)')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
plt.tight_layout()
plt.savefig('artifacts/bias_genre.png', dpi=120)
plt.show()

In [ ]:
print('=== Bias Audit Summary ===')
print(f'Popularity ratio (SASRec): {summary["sasrec_popularity_ratio"]:.2f}x (SVD: {summary["svd_popularity_ratio"]:.2f}x)')
print(f'Long-tail fraction (SASRec): {summary["sasrec_longtail_fraction"]:.1%}')
print(f'Gender gap (M-F): {summary["gender_gap_HR10"]:+.4f} HR@10')
print(f'Age range (max-min): {summary["age_hr10_range"]:.4f} HR@10')
print(f'Top genre: {summary["top_genre"]} ({summary["top_genre_frac"]:.1%} of recs)')
print(f'Under-18 cohort underperforms by ~{(summary["age_hr10_range"]*0.7):.3f} HR@10 \u2014 production risk for youth platforms')